# 001. Collect ADS records

Canonical ADS retrieval stage for the paper pipeline.


## Purpose, inputs, and outputs

**Purpose**
- Query NASA ADS for records containing the phrase `"dark matter"`.
- Hydrate missing abstracts.
- Fetch citation histories from `v1/metrics/detail` with `types=["citations"]`.
- Retain a Git-friendly stage-001 snapshot bundle for the public transparency repository.

**Requires**
- An ADS API token in `ADS_TOKEN`, or pasted into `USER_ADS_TOKEN` in the setup cell.

**Writes**
- `code/stage-outputs/001-collect-ads-records/ads_search_all_years.json` (local raw snapshot)
- `code/stage-outputs/001-collect-ads-records/ads_search_all_years_with_abstracts.json` (retained snapshot)
- `code/stage-outputs/001-collect-ads-records/ads_search_all_years_metrics_citations.json`
- `code/stage-outputs/001-collect-ads-records/ads_stage_001_snapshots.zip`


In [ ]:
import os

from shared.ads_api import (
    DEFAULT_METRICS_TYPES,
    DEFAULT_SEARCH_FIELDS,
    build_phrase_query,
    build_zip_archive,
    fetch_metrics_for_bibcodes,
    fetch_missing_abstracts,
    load_json,
    query_ads_api,
    save_json,
    set_ads_token,
)
from shared.project_paths import get_project_paths

paths = get_project_paths()
output_dir = paths.output_dir("001-collect-ads-records")

SEARCH_FIELDS = DEFAULT_SEARCH_FIELDS
METRIC_TYPES = DEFAULT_METRICS_TYPES
SEARCH_PHRASE = "dark matter"
SEARCH_YEAR = None                      # Set to an integer year only if you want data for a specific year.
USER_ADS_TOKEN = ""                     # Optional: paste your ADS token here for this session only.

if USER_ADS_TOKEN.strip():
    set_ads_token(USER_ADS_TOKEN)

ADS_TOKEN = os.environ.get("ADS_TOKEN", "")

output_stem = "ads_search_all_years" if SEARCH_YEAR is None else f"ads_search_{SEARCH_YEAR}"
RECORDS_JSON = output_dir / f"{output_stem}.json"
HYDRATED_JSON = output_dir / f"{output_stem}_with_abstracts.json"
METRICS_JSON = output_dir / f"{output_stem}_metrics_citations.json"
SNAPSHOT_ARCHIVE = output_dir / ("ads_stage_001_snapshots.zip" if SEARCH_YEAR is None else f"{output_stem}_snapshots.zip")

print("Records JSON:", RECORDS_JSON)
print("Hydrated JSON:", HYDRATED_JSON)
print("Metrics JSON:", METRICS_JSON)
print("Snapshot archive:", SNAPSHOT_ARCHIVE)
print("Using ADS token from environment:", bool(ADS_TOKEN))
print("Search year restriction:", SEARCH_YEAR)
print("Search fields:", SEARCH_FIELDS)
print("Metrics types:", METRIC_TYPES)


## Stage 1. Query ADS for the record set


In [ ]:
if not ADS_TOKEN:
    raise ValueError(
        "Set ADS_TOKEN in your environment, or paste it into USER_ADS_TOKEN in the setup cell."
    )

encoded_query = build_phrase_query(phrase=SEARCH_PHRASE, year=SEARCH_YEAR, fields=SEARCH_FIELDS)
records = query_ads_api(encoded_query, ADS_TOKEN)
save_json(records, RECORDS_JSON)

scope = "all years" if SEARCH_YEAR is None else str(SEARCH_YEAR)
print(f"Saved {len(records):,} ADS records for {scope} to {RECORDS_JSON}")


## Stage 2. Fill missing abstracts


In [ ]:
records = load_json(RECORDS_JSON, default=[])
hydrated_records = fetch_missing_abstracts(records, ADS_TOKEN)
save_json(hydrated_records, HYDRATED_JSON)

missing_after = sum(1 for record in hydrated_records if not record.get("abstract"))
print(f"Saved hydrated ADS records to {HYDRATED_JSON}")
print(f"Records still missing abstracts: {missing_after:,}")


## Stage 3. Fetch citation histories


In [ ]:
hydrated_records = load_json(HYDRATED_JSON, default=[])
bibcodes = [record.get("bibcode") for record in hydrated_records if record.get("bibcode")]
metrics_citations = fetch_metrics_for_bibcodes(
    bibcodes,
    ADS_TOKEN,
    metric_types=METRIC_TYPES,
)
save_json(metrics_citations, METRICS_JSON)
build_zip_archive([HYDRATED_JSON, METRICS_JSON], SNAPSHOT_ARCHIVE)

print(f"Saved citation metrics payload for {len(bibcodes):,} bibcodes to {METRICS_JSON}")
print(f"Saved retained stage-001 snapshot archive to {SNAPSHOT_ARCHIVE}")


## Stage 4. Inspect the canonical outputs


In [ ]:
if HYDRATED_JSON.exists() and METRICS_JSON.exists():
    hydrated_records = load_json(HYDRATED_JSON, default=[])
    metrics_citations = load_json(METRICS_JSON, default={})
    print(f"Hydrated record count: {len(hydrated_records):,}")
    print(f"Metrics payload count: {len(metrics_citations):,}")
    print("Snapshot archive:", SNAPSHOT_ARCHIVE)

    for i, record in enumerate(hydrated_records[:3], 1):
        print(f"\nRecord {i}")
        print("bibcode:", record.get("bibcode"))
        print("year:", record.get("year"))
        abstract = record.get("abstract") or ""
        print("abstract preview:", abstract[:240])
else:
    print("Run the retrieval, hydration, and metrics cells first.")
